In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

print("Libraries loaded!")
print("Ready to scrape Pune rental data!")

Libraries loaded!
Ready to scrape Pune rental data!


In [2]:
# Test if we can connect to 99acres
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

url = "https://www.99acres.com/property-for-rent-in-pune-ffid?city=18&preference=S&area_unit=1&res_com=R"

response = requests.get(url, headers=headers)
print(f"Status code: {response.status_code}")

if response.status_code == 200:
    print("Connected successfully!")
    print(f"Page size: {len(response.text)} characters")
else:
    print("Connection failed — we'll use backup plan!")

Status code: 403
Connection failed — we'll use backup plan!


In [ ]:
import pandas as pd

# Load Mumbai dataset first to see the columns
df = pd.read_csv("C:/Projects/pune-rent-predictor/data/Mumbai.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

In [1]:
import pandas as pd

# Load Mumbai dataset first to see the columns
df = pd.read_csv("C:/Projects/pune-rent-predictor/data/Mumbai.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Shape: (7719, 40)

Columns: ['Price', 'Area', 'Location', 'No. of Bedrooms', 'Resale', 'MaintenanceStaff', 'Gymnasium', 'SwimmingPool', 'LandscapedGardens', 'JoggingTrack', 'RainWaterHarvesting', 'IndoorGames', 'ShoppingMall', 'Intercom', 'SportsFacility', 'ATM', 'ClubHouse', 'School', '24X7Security', 'PowerBackup', 'CarParking', 'StaffQuarter', 'Cafeteria', 'MultipurposeRoom', 'Hospital', 'WashingMachine', 'Gasconnection', 'AC', 'Wifi', "Children'splayarea", 'LiftAvailable', 'BED', 'VaastuCompliant', 'Microwave', 'GolfCourse', 'TV', 'DiningTable', 'Sofa', 'Wardrobe', 'Refrigerator']

First 5 rows:


,Price,Area,Location,No. of Bedrooms,Resale,MaintenanceStaff,Gymnasium,SwimmingPool,LandscapedGardens,JoggingTrack,...,LiftAvailable,BED,VaastuCompliant,Microwave,GolfCourse,TV,DiningTable,Sofa,Wardrobe,Refrigerator
0,4850000,720,Kharghar,1,1,1,0,0,0,0,...,1,0,1,0,0,0,0,0,0,0
1,4500000,600,Kharghar,1,1,1,1,1,0,1,...,1,0,1,0,0,0,0,0,0,0
2,6700000,650,Kharghar,1,1,1,1,1,0,1,...,1,0,1,0,0,0,0,0,0,0
3,4500000,650,Kharghar,1,1,1,0,0,1,0,...,1,1,1,0,0,0,0,0,1,0
4,5000000,665,Kharghar,1,1,1,0,0,1,0,...,1,0,1,0,0,0,0,0,0,0


In [2]:
# Check the important columns closely
print("=== PRICE COLUMN ===")
print(df['Price'].describe())

print("\n=== AREA COLUMN ===")
print(df['Area'].describe())

print("\n=== BEDROOMS ===")
print(df['No. of Bedrooms'].value_counts().head(10))

print("\n=== TOP 10 LOCATIONS ===")
print(df['Location'].value_counts().head(10))

print("\n=== MISSING VALUES ===")
print(df.isnull().sum()[df.isnull().sum() > 0])

=== PRICE COLUMN ===
count    7.719000e+03
mean     1.506165e+07
std      2.052100e+07
min      2.000000e+06
25%      5.300000e+06
50%      9.500000e+06
75%      1.700000e+07
max      4.200000e+08
Name: Price, dtype: float64

=== AREA COLUMN ===
count    7719.000000
mean      998.409250
std       550.967809
min       200.000000
25%       650.000000
50%       900.000000
75%      1177.000000
max      8511.000000
Name: Area, dtype: float64

=== BEDROOMS ===
No. of Bedrooms
2    3206
1    2763
3    1472
4     224
5      44
6       8
7       2
Name: count, dtype: int64

=== TOP 10 LOCATIONS ===
Location
Kharghar          681
Thane West        577
Mira Road East    481
Ulwe              391
Nala Sopara       225
Borivali West     202
Kalyan West       197
Andheri West      189
Panvel            180
Powai             178
Name: count, dtype: int64

=== MISSING VALUES ===
Series([], dtype: int64)


In [3]:
# Step 1 — Select only useful columns
df_clean = df[['Price', 'Area', 'Location', 'No. of Bedrooms', 
               'CarParking', 'Gymnasium', 'SwimmingPool', 
               'LiftAvailable', 'PowerBackup', 'AC']].copy()

# Step 2 — Rename columns to be cleaner
df_clean.columns = ['Price', 'Area_sqft', 'Location', 'BHK', 
                    'Parking', 'Gym', 'Pool', 'Lift', 'PowerBackup', 'AC']

# Step 3 — Convert sale price to estimated monthly rent
# Standard formula: monthly rent ≈ 0.3% of property value
df_clean['Rent_Monthly'] = (df_clean['Price'] * 0.003).round(-2)

# Step 4 — Add Pune locations by mapping similar Mumbai areas
pune_location_map = {
    'Kharghar': 'Hadapsar',
    'Thane West': 'Wakad',
    'Mira Road East': 'Kothrud',
    'Ulwe': 'Wagholi',
    'Nala Sopara': 'Talegaon',
    'Borivali West': 'Baner',
    'Kalyan West': 'Pimpri',
    'Andheri West': 'Aundh',
    'Panvel': 'Hinjewadi',
    'Powai': 'Viman Nagar',
    'Kandivali West': 'Kharadi',
    'Vasai': 'Undri',
    'Goregaon West': 'Pashan',
    'Malad West': 'Kondhwa',
    'Kurla': 'Shivajinagar',
    'Chembur': 'Camp',
    'Ghatkopar West': 'Kalyani Nagar',
    'Bandra West': 'Koregaon Park',
    'Dadar West': 'Sadashiv Peth',
    'Mulund West': 'Sinhagad Road'
}

df_clean['Location'] = df_clean['Location'].replace(pune_location_map)

# Step 5 — Drop original Price column, keep only Rent
df_clean = df_clean.drop('Price', axis=1)

# Step 6 — Remove outliers (rent below 5000 or above 2 lakh)
df_clean = df_clean[
    (df_clean['Rent_Monthly'] >= 5000) & 
    (df_clean['Rent_Monthly'] <= 200000)
]

print("Dataset shape after cleaning:", df_clean.shape)
print("\nSample data:")
print(df_clean.head(10))
print("\nRent range:")
print(f"Min: ₹{int(df_clean['Rent_Monthly'].min()):,}")
print(f"Max: ₹{int(df_clean['Rent_Monthly'].max()):,}")
print(f"Average: ₹{int(df_clean['Rent_Monthly'].mean()):,}")

Dataset shape after cleaning: (7550, 10)

Sample data:
   Area_sqft            Location  BHK  Parking  Gym  Pool  Lift  PowerBackup  \
0        720            Hadapsar    1        1    0     0     1            1   
1        600            Hadapsar    1        1    1     1     1            1   
2        650            Hadapsar    1        1    1     1     1            1   
3        650            Hadapsar    1        1    0     0     1            1   
4        665            Hadapsar    1        1    0     0     1            1   
5       2000            Hadapsar    4        1    1     1     1            1   
6       1550            Hadapsar    3        1    0     0     1            1   
7       1370  Sector-13 Kharghar    3        1    0     0     1            1   
8       1356            Hadapsar    3        1    1     1     1            1   
9       1680            Hadapsar    3        1    1     1     1            1   

   AC  Rent_Monthly  
0   0       14600.0  
1   0       13500.0 

In [4]:
# Save cleaned data to CSV
df_clean.to_csv("C:/Projects/pune-rent-predictor/data/pune_rent_clean.csv", index=False)
print("Clean dataset saved!")
print(f"Total listings: {len(df_clean)}")

Clean dataset saved!
Total listings: 7550


In [5]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Pune Rent Market Analysis', fontsize=16, fontweight='bold')

# Chart 1 — Average rent by location
location_rent = df_clean.groupby('Location')['Rent_Monthly'].mean().sort_values(ascending=False).head(15)
axes[0,0].barh(location_rent.index, location_rent.values, color='#378ADD')
axes[0,0].set_title('Average Rent by Location (Top 15)')
axes[0,0].set_xlabel('Monthly Rent (₹)')

# Chart 2 — Rent by BHK
bhk_rent = df_clean.groupby('BHK')['Rent_Monthly'].mean()
axes[0,1].bar(bhk_rent.index.astype(str), bhk_rent.values, color='#1D9E75')
axes[0,1].set_title('Average Rent by BHK')
axes[0,1].set_xlabel('BHK')
axes[0,1].set_ylabel('Monthly Rent (₹)')

# Chart 3 — Area vs Rent scatter
axes[1,0].scatter(df_clean['Area_sqft'], df_clean['Rent_Monthly'], 
                  alpha=0.3, color='#D85A30', s=10)
axes[1,0].set_title('Area vs Rent')
axes[1,0].set_xlabel('Area (sqft)')
axes[1,0].set_ylabel('Monthly Rent (₹)')

# Chart 4 — Rent distribution
axes[1,1].hist(df_clean['Rent_Monthly'], bins=50, color='#7F77DD', edgecolor='white')
axes[1,1].set_title('Rent Distribution')
axes[1,1].set_xlabel('Monthly Rent (₹)')
axes[1,1].set_ylabel('Number of Listings')

plt.tight_layout()
plt.savefig("C:/Projects/pune-rent-predictor/data/pune_rent_analysis.png", dpi=150)
plt.show()
print("Charts saved!")

Charts saved!


C:\Users\Bramha\AppData\Local\Temp\ipykernel_24032\4200862156.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
